# Quant Trading DBMS Analysis

This notebook visualizes the performance of the Mean Reversion trading strategy backtested over the generated mock stock data. It connects directly to the resulting SQLite database to extract trades and portfolio performance.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Prettify plots
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (14, 7)

# Connect to local database
conn = sqlite3.connect('../database/quant_trading.db')

## 1. Equity Curve
The equity curve shows the daily value of the portfolio cash plus marked-to-market positions over time.

In [ ]:
query = "SELECT date, portfolio_value FROM portfolio_performance ORDER BY date ASC"
portfolio_df = pd.read_sql(query, conn, parse_dates=['date'])

if len(portfolio_df) > 0:
    plt.figure(figsize=(14, 6))
    plt.plot(portfolio_df['date'], portfolio_df['portfolio_value'], label='Portfolio Value (Cash + Holdings)', color='royalblue')
    plt.title('Equity Curve')
    plt.xlabel('Date')
    plt.ylabel('Value ($)')
    plt.legend()
    plt.show()
else:
    print("No portfolio data found. Did you run `main.py` first?")

## 2. Drawdown Chart
Drawdown shows the percentage drop from the highest peak of the portfolio to the current value.

In [ ]:
if len(portfolio_df) > 0:
    portfolio_df['peak'] = portfolio_df['portfolio_value'].cummax()
    portfolio_df['drawdown'] = (portfolio_df['portfolio_value'] - portfolio_df['peak']) / portfolio_df['peak']
    
    plt.figure(figsize=(14, 4))
    plt.fill_between(portfolio_df['date'], portfolio_df['drawdown'], 0, color='red', alpha=0.3)
    plt.plot(portfolio_df['date'], portfolio_df['drawdown'], color='red')
    plt.title('Portfolio Drawdown')
    plt.xlabel('Date')
    plt.ylabel('Drawdown percentage')
    plt.show()

## 3. Trade Signals overlaying Stock Price
Let's visualize the stock price along with the points where the strategy issued BUY (Green) and SELL (Red) signals.

In [ ]:
# Example for AAPL
ticker_to_analyze = 'AAPL'

price_query = f"""
SELECT p.date, p.close
FROM price_data p 
JOIN stocks s ON p.stock_id = s.stock_id
WHERE s.ticker = '{ticker_to_analyze}'
ORDER BY p.date ASC
"""

signals_query = f"""
SELECT ts.date, ts.signal_type
FROM trading_signals ts  
JOIN stocks s ON ts.stock_id = s.stock_id
WHERE s.ticker = '{ticker_to_analyze}' AND ts.signal_type IN ('BUY', 'SELL')
"""

try:
    price_df = pd.read_sql(price_query, conn, parse_dates=['date'])
    signals_df = pd.read_sql(signals_query, conn, parse_dates=['date'])
    
    if len(price_df) > 0:
        plt.figure(figsize=(14, 7))
        plt.plot(price_df['date'], price_df['close'], label=f'{ticker_to_analyze} Close Price', alpha=0.6, color='grey')
        
        # Merge to get the price on signal dates
        merged = pd.merge(signals_df, price_df, on='date', how='inner')
        
        buys = merged[merged['signal_type'] == 'BUY']
        sells = merged[merged['signal_type'] == 'SELL']
        
        plt.scatter(buys['date'], buys['close'], marker='^', color='green', s=100, label='BUY Signal', zorder=5)
        plt.scatter(sells['date'], sells['close'], marker='v', color='red', s=100, label='SELL Signal', zorder=5)
        
        plt.title(f'{ticker_to_analyze} Price alongside Generated Strategy Signals')
        plt.xlabel('Date')
        plt.ylabel('Price')
        plt.legend()
        plt.show()

except pd.errors.DatabaseError:
    print("Database error or empty tables. Ensure main.py finished execution.")

# Close Database connection gracefully
conn.close()